# Rerun W&B Run Plot

Focused notebook backed by `wandb_metrics.py` for GINE best_00 rerun comparisons and A0 rerun comparisons.


In [ ]:
from pathlib import Path
import importlib
import sys

for candidate in [Path.cwd(), *Path.cwd().parents]:
    helpers = candidate / "helpers"
    repo_helpers = candidate / "Topology_Task" / "analysis" / "metrics" / "helpers"
    if helpers.exists() and (helpers / "wandb_metrics.py").exists():
        sys.path.insert(0, str(helpers))
        break
    if repo_helpers.exists() and (repo_helpers / "wandb_metrics.py").exists():
        sys.path.insert(0, str(repo_helpers))
        break
else:
    raise FileNotFoundError("Could not locate Topology_Task/analysis/metrics/helpers")

import wandb_metrics as wm
wm = importlib.reload(wm)
print("wandb_metrics:", wm.__file__)


## Choose Runs To Load Or Download


In [ ]:
import importlib
wm = importlib.reload(wm)

# Change this if you only want a subset of seeds.
SEEDS = (0, 1, 2)

# Pick presets, config folder(s), exact W&B run names, or use None / "None" / "all" for every run.
# Presets:
#   "gine_rerun" -> GINE best_00 plus best_10, best_11, best_12, best_13, best_14
#   "a0_rerun"   -> rerun_a0_opt, rerun_a0_nonopt, rerun_a01_opt, rerun_a02_opt, a0_nonopt, rerun_bias_05, rerun_bias_07
#   "a0_bias_rerun" -> rerun_bias_05 and rerun_bias_07 only
# Examples:
# RUN_SELECTIONS_TO_DOWNLOAD = "gine_rerun"
# RUN_SELECTIONS_TO_DOWNLOAD = ["gine_rerun", "a0_rerun"]
# RUN_SELECTIONS_TO_DOWNLOAD = "gine_rerun,a0_rerun"
# RUN_SELECTIONS_TO_DOWNLOAD = ["gine_s0_s1_s2", "a0_test_rerun"]
# RUN_SELECTIONS_TO_DOWNLOAD = ["a0_rerun", "a0_bias_rerun"]
# RUN_SELECTIONS_TO_DOWNLOAD = "all"
RUN_SELECTIONS_TO_DOWNLOAD = ["gine_rerun", "a0_rerun"]

# False only reads local cached histories. True downloads missing matching histories from W&B.
DOWNLOAD_MISSING_FROM_WANDB = True

# Turn this on only when W&B has newer data than a stale local scan_history fallback cache.
# This only has an effect when DOWNLOAD_MISSING_FROM_WANDB is True.
REFRESH_SCAN_HISTORY_FALLBACKS = False

# Heavier option: replace every selected local cache from W&B.
FORCE_REFRESH_CACHE = False

RUN_NAMES_TO_DOWNLOAD = wm.resolve_rerun_wandb_run_selection(RUN_SELECTIONS_TO_DOWNLOAD, seeds=SEEDS)
wm.configure_run_filter_from_rerun_selection(RUN_SELECTIONS_TO_DOWNLOAD, seeds=SEEDS)
wm.EXCLUDE_RUN_NAME_REGEX = None
wm.RUN_STATES = None
wm.MAX_RUNS = None
wm.USE_LOCAL_CACHE_ONLY = not DOWNLOAD_MISSING_FROM_WANDB
wm.REFRESH_SCAN_HISTORY_FALLBACKS = bool(DOWNLOAD_MISSING_FROM_WANDB and REFRESH_SCAN_HISTORY_FALLBACKS)
wm.FORCE_REFRESH = bool(DOWNLOAD_MISSING_FROM_WANDB and FORCE_REFRESH_CACHE)
wm.ALLOW_SCAN_HISTORY_FALLBACK = True
wm.refresh_run_filters()

if RUN_NAMES_TO_DOWNLOAD is None:
    print("Selected run-name candidates: all W&B runs")
else:
    print("Selected run-name candidates:", len(RUN_NAMES_TO_DOWNLOAD))
print("RUN_NAME_REGEX =", wm.RUN_NAME_REGEX)
print("USE_LOCAL_CACHE_ONLY =", wm.USE_LOCAL_CACHE_ONLY)
print("REFRESH_SCAN_HISTORY_FALLBACKS =", wm.REFRESH_SCAN_HISTORY_FALLBACKS)
print("FORCE_REFRESH =", wm.FORCE_REFRESH)


## Load Selected W&B Histories


In [ ]:
data = wm.load_wandb_data()

runs_df = data.runs_df
history_df = data.history_df
runs_df


## GINE best_00 vs best_10-14


- 00 - no opt; bias 0.7; entropy 0.02; concat; heavy GINE
- 10 - opt; bias 0.7; entropy 0.02; concat; light GINE
- 11 - no opt; bias 0; entropy 0.02; concat; heavy GINE
- 12 - opt; bias 0; entropy 0.02; concat; light GINE
- 13 - opt; bias 0; entropy 0.01; concat; light GINE
- 14 - opt; bias 0; entropy 0.01; no concat; light GINE


In [ ]:
gine_rerun_result = wm.plot_rerun_gine_best00_comparisons(seeds=SEEDS)
gine_rerun_result["fig"]


## A0 Rerun Comparison


- a0 - opt; bias 0; reward norm; [256, 256, 256]
- a0 nonopt - no optimized critic version of the same rerun family
- a01 - opt; bias 0; reward norm; [256, 256]
- a02 - opt; bias 0; no reward norm; [256, 256, 256]
- old a0_nonopt - deterministic no-entropy-decay nonopt control


In [ ]:
a0_rerun_result = wm.plot_rerun_a0_opt_comparison(seeds=SEEDS)
a0_rerun_result["fig"]


## A0 Bias Comparison


In [ ]:
a0_bias_result = wm.plot_rerun_a0_bias_comparison(seeds=SEEDS)
a0_bias_result["fig"]
